# Hyper-parameter search

fANOVA importance of each parameter, and every trial's success against k.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

import utils as u

FS_SMALL, FS_TINY = 9, 7  # pt: \small, \scriptsize of the paper
PAGE_WIDTH = 5.5  # in: \textwidth of the paper (\linewidth of a figure)
ROOT = Path.cwd().parent
data_path = ROOT / "results"
figs_path = ROOT / "analysis/figs"
style_path = ROOT / "analysis/figure.mplstyle"

In [ ]:
data = json.loads((data_path / "data.json").read_text())
figs_path.mkdir(exist_ok=True)
plt.style.use(style_path)

In [ ]:
OPTUNA_NAMES = {"k": "k (neighbours)", "one_per_episode": "≤ 1 neighbour per demo", "chunk_size": "chunk length",
                "use_consistency": "trajectory stickiness", "w_wrist": "wrist-camera weight",
                "temperature": "softmax temperature", "temporal_ensemble": "temporal ensembling",
                "distance_normalization": "distance normalisation", "w_agentview": "agent-view weight",
                "norm_preserving": "norm preserving", "use_history": "history matching", "use_action_mode": "mode seeking",
                "task_conditioning": "prompt conditioning", "state_weight": "state weight", "time_weight": "time weight",
                "use_time": "time feature", "memory_stride": "memory stride", "image_encoder": "image encoder",
                "pooling_resnet18": "feature pooling", "ensemble_coeff": "ensembling decay",
                "use_ensemble_mode": "ensemble mode seeking", "gripper_mode": "gripper averaging"}  # fmt: skip

In [ ]:
op = data["optuna"]
imp = list(op["importance"].items())[:8][::-1]
trials = [t for t in op["trials"] if "k" in t["params"]]

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(PAGE_WIDTH, 2.232), layout="constrained", gridspec_kw={"width_ratios": [1.15, 1]})
ax1.barh(range(len(imp)), [100 * v for _k, v in imp], color=u.Colour.KNN, height=0.62, edgecolor="white")
ax1.set_yticks(range(len(imp)), [OPTUNA_NAMES.get(k, k) for k, _v in imp])
for i, (_k, v) in enumerate(imp):
    ax1.text(100 * v + 1, i, f"{100 * v:.0f}%", va="center", fontsize=FS_TINY)
ax1.set_xlabel("importance (fANOVA)", fontsize=FS_SMALL)
ax1.set_xlim(0, 75)
ax1.tick_params(axis="y", length=0)
ax1.spines["left"].set_visible(False)
ax1.xaxis.grid(True)
ax1.set_axisbelow(True)
for one, colour, lab in ((True, u.Colour.KNN, "≤ 1 per demo"), (False, u.Colour.HUMAN, "any frames")):
    sel = [t for t in trials if bool(t["params"].get("one_per_episode", True)) == one]
    ax2.scatter([t["params"]["k"] for t in sel], [100 * t["value"] for t in sel], s=34, color=colour,
              edgecolor="white", linewidth=0.8, label=lab, zorder=3)  # fmt: skip
ax2.set_xlabel("k", fontsize=FS_SMALL)
ax2.set_ylabel("trial success (%)", fontsize=FS_SMALL)
ax2.grid(True)
ax2.set_axisbelow(True)
ax2.legend(frameon=False, fontsize=FS_TINY, loc="lower right")
fig.savefig(figs_path / "optuna.pdf")